In [1]:
from google.colab import drive
drive.mount('/content/drive')

# Path to your zip file
zip_path = "/content/drive/MyDrive/archive.zip"

# Unzip
!unzip -q "$zip_path" -d /content/

Mounted at /content/drive


In [2]:
import os

for root, dirs, files in os.walk('/content'):
    if 'train' in dirs:
        print("Found train folder at:", os.path.join(root, 'train'))
    if 'validation' in dirs:
        print("Found validation folder at:", os.path.join(root, 'validation'))

Found train folder at: /content/images/train
Found validation folder at: /content/images/validation
Found train folder at: /content/images/images/train
Found validation folder at: /content/images/images/validation


In [3]:
TRAIN_DIR = "/content/images/train"
TEST_DIR  = "/content/images/validation"

In [4]:
import pandas as pd

def createdataframe(dir_path):
    image_paths = []
    labels = []

    for label in os.listdir(dir_path):
        label_path = os.path.join(dir_path, label)
        if not os.path.isdir(label_path):
            continue
        for img in os.listdir(label_path):
            image_paths.append(os.path.join(label_path, img))
            labels.append(label)

    return image_paths, labels

train = pd.DataFrame()
train['image'], train['label'] = createdataframe(TRAIN_DIR)

test = pd.DataFrame()
test['image'], test['label'] = createdataframe(TEST_DIR)

print(train.head())

                                  image label
0    /content/images/train/fear/978.jpg  fear
1  /content/images/train/fear/26679.jpg  fear
2   /content/images/train/fear/7605.jpg  fear
3  /content/images/train/fear/26520.jpg  fear
4    /content/images/train/fear/723.jpg  fear


In [5]:
import numpy as np
from tensorflow.keras.utils import load_img, img_to_array
from tqdm import tqdm

def extract_features(images, target_size=(48,48)):
    features = []
    for image in tqdm(images):
        img = load_img(image, color_mode='grayscale', target_size=target_size)
        img = img_to_array(img)
        features.append(img)

    features = np.array(features, dtype='float32')
    features /= 255.0
    return features

x_train = extract_features(train['image'])
x_test  = extract_features(test['image'])

100%|██████████| 7066/7066 [00:00<00:00, 7322.91it/s]


In [6]:
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.utils import to_categorical

le = LabelEncoder()

y_train = le.fit_transform(train['label'])
y_test  = le.transform(test['label'])

y_train = to_categorical(y_train, num_classes=7)
y_test  = to_categorical(y_test, num_classes=7)

In [7]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, BatchNormalization
from tensorflow.keras.layers import Flatten, Dense, Dropout, Input

model = Sequential()

model.add(Input(shape=(48,48,1)))

# Block 1
model.add(Conv2D(64, (3,3), padding='same', activation='relu'))
model.add(BatchNormalization())
model.add(Conv2D(64, (3,3), padding='same', activation='relu'))
model.add(MaxPooling2D((2,2)))
model.add(Dropout(0.2))

# Block 2
model.add(Conv2D(128, (3,3), padding='same', activation='relu'))
model.add(BatchNormalization())
model.add(Conv2D(128, (3,3), padding='same', activation='relu'))
model.add(MaxPooling2D((2,2)))
model.add(Dropout(0.25))

# Block 3
model.add(Conv2D(256, (3,3), padding='same', activation='relu'))
model.add(BatchNormalization())
model.add(Conv2D(256, (3,3), padding='same', activation='relu'))
model.add(MaxPooling2D((2,2)))
model.add(Dropout(0.3))

# Dense
model.add(Flatten())

model.add(Dense(256, activation='relu'))
model.add(BatchNormalization())
model.add(Dropout(0.5))

model.add(Dense(128, activation='relu'))
model.add(Dropout(0.4))

# Output
model.add(Dense(7, activation='softmax'))

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 48, 48, 64)     │           640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 48, 48, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 48, 48, 64)     │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 24, 24, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 24, 24, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 24, 24, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 24, 24, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 24, 24, 128)    │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 12, 12, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 12, 12, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 12, 12, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 12, 12, 256)    │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 12, 12, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 6, 6, 256)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 6, 6, 256)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 9216)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │     2,359,552 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 256)            │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 7)              │           903 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,540,423 (13.51 MB)

 Trainable params: 3,539,015 (13.50 MB)

 Non-trainable params: 1,408 (5.50 KB)

In [8]:
history = model.fit(
    x_train, y_train,
    batch_size=128,
    epochs=25,
    validation_data=(x_test, y_test)
)

Epoch 1/25
226/226 ━━━━━━━━━━━━━━━━━━━━ 46s 114ms/step - accuracy: 0.2706 - loss: 1.9282 - val_accuracy: 0.2586 - val_loss: 1.9465
Epoch 2/25
226/226 ━━━━━━━━━━━━━━━━━━━━ 14s 64ms/step - accuracy: 0.4021 - loss: 1.5523 - val_accuracy: 0.4186 - val_loss: 1.6138
Epoch 3/25
226/226 ━━━━━━━━━━━━━━━━━━━━ 15s 65ms/step - accuracy: 0.4777 - loss: 1.3573 - val_accuracy: 0.4505 - val_loss: 1.4396
Epoch 4/25
226/226 ━━━━━━━━━━━━━━━━━━━━ 15s 66ms/step - accuracy: 0.5203 - loss: 1.2628 - val_accuracy: 0.5501 - val_loss: 1.1936
Epoch 5/25
226/226 ━━━━━━━━━━━━━━━━━━━━ 15s 67ms/step - accuracy: 0.5490 - loss: 1.1884 - val_accuracy: 0.5454 - val_loss: 1.2048
Epoch 6/25
226/226 ━━━━━━━━━━━━━━━━━━━━ 15s 67ms/step - accuracy: 0.5645 - loss: 1.1417 - val_accuracy: 0.5512 - val_loss: 1.1934
Epoch 7/25
226/226 ━━━━━━━━━━━━━━━━━━━━ 15s 68ms/step - accuracy: 0.5930 - loss: 1.0851 - val_accuracy: 0.5800 - val_loss: 1.1062
Epoch 8/25
226/226 ━━━━━━━━━━━━━━━━━━━━ 15s 67ms/step - accuracy: 0.6087 - loss: 1.0404 -

In [9]:
model.save("/content/drive/MyDrive/emotiondetector.h5")

print("Model saved successfully!")

Model saved successfully!


In [10]:
from tensorflow.keras.models import load_model

model = load_model("/content/drive/MyDrive/emotiondetector.h5")

print("Model loaded!")

Model loaded!


In [11]:
emotion_labels = ['angry','disgust','fear','happy','sad','surprise','neutral']

In [ ]:
from google.colab import files
uploaded = files.upload()

TypeError: 'NoneType' object is not subscriptable

In [ ]:
import numpy as np
from tensorflow.keras.utils import load_img, img_to_array
import matplotlib.pyplot as plt

# Get uploaded image name
img_path = list(uploaded.keys())[0]

# Load and preprocess
img = load_img(img_path, color_mode='grayscale', target_size=(48,48))
img_array = img_to_array(img)
img_array = img_array / 255.0
img_array = np.reshape(img_array, (1,48,48,1))

# Predict
prediction = model.predict(img_array)
predicted_label = emotion_labels[np.argmax(prediction)]

# Show result
plt.imshow(img, cmap='gray')
plt.title(f"Predicted Emotion: {predicted_label}")
plt.axis('off')
plt.show()


In [ ]:
loss, accuracy = model.evaluate(x_test, y_test)
print(f"Test Accuracy: {accuracy*100:.2f}%")

In [ ]:
import matplotlib.pyplot as plt

# Accuracy
plt.plot(history.history['accuracy'], label='train accuracy')
plt.plot(history.history['val_accuracy'], label='val accuracy')
plt.legend()
plt.title("Accuracy")
plt.show()

# Loss
plt.plot(history.history['loss'], label='train loss')
plt.plot(history.history['val_loss'], label='val loss')
plt.legend()
plt.title("Loss")
plt.show()

In [ ]:
from IPython.display import display, Javascript
from google.colab.output import eval_js
from base64 import b64decode

def take_photo(filename='photo.jpg', quality=0.8):
    js = Javascript('''
        async function takePhoto(quality) {
            const div = document.createElement('div');
            const capture = document.createElement('button');
            capture.textContent = '📸 Capture';
            div.appendChild(capture);

            const video = document.createElement('video');
            video.style.display = 'block';
            const stream = await navigator.mediaDevices.getUserMedia({video: true});

            document.body.appendChild(div);
            div.appendChild(video);
            video.srcObject = stream;
            await video.play();

            await new Promise((resolve) => capture.onclick = resolve);

            const canvas = document.createElement('canvas');
            canvas.width = video.videoWidth;
            canvas.height = video.videoHeight;
            canvas.getContext('2d').drawImage(video, 0, 0);

            stream.getTracks().forEach(track => track.stop());
            div.remove();

            return canvas.toDataURL('image/jpeg', quality);
        }
    ''')

    display(js)
    data = eval_js('takePhoto({})'.format(quality))
    binary = b64decode(data.split(',')[1])

    with open(filename, 'wb') as f:
        f.write(binary)

    return filename

In [ ]:
img_path = take_photo()
print("Saved to:", img_path)

In [ ]:
import cv2

# Download Haar Cascade file (only once)
!wget -q https://github.com/opencv/opencv/raw/master/data/haarcascades/haarcascade_frontalface_default.xml

# Load classifier
face_cascade = cv2.CascadeClassifier("haarcascade_frontalface_default.xml")

In [ ]:
import numpy as np

emotion_labels = ['angry','disgust','fear','happy','sad','surprise','neutral']

img = cv2.imread(img_path)
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

faces = face_cascade.detectMultiScale(gray, 1.3, 5)

print("Faces detected:", len(faces))

for (x,y,w,h) in faces:
    face = gray[y:y+h, x:x+w]

    face = cv2.resize(face, (48,48))
    face = face / 255.0
    face = face.reshape(1,48,48,1)

    pred = model.predict(face)
    label = emotion_labels[np.argmax(pred)]

    cv2.rectangle(img,(x,y),(x+w,y+h),(0,255,0),2)
    cv2.putText(img,label,(x,y-10),
                cv2.FONT_HERSHEY_SIMPLEX,1,(0,255,0),2)

import matplotlib.pyplot as plt
plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
plt.axis('off')
plt.show()

In [ ]:
!pip install deepface
!pip install face_recognition


In [ ]:
import cv2
import numpy as np
import pandas as pd
import face_recognition

from deepface import DeepFace
from datetime import datetime
from google.colab.patches import cv2_imshow


In [ ]:
attendance_file = "attendance.csv"

try:
    attendance_df = pd.read_csv(attendance_file)
except:
    attendance_df = pd.DataFrame(
        columns=["Name", "Date", "Time", "Emotion"]
    )

In [ ]:
from google.colab import files

uploaded = files.upload()

# Get uploaded filename automatically
image_name = list(uploaded.keys())[0]

print("Uploaded File:", image_name)

In [ ]:
known_image = face_recognition.load_image_file(image_name)

known_encoding = face_recognition.face_encodings(
    known_image
)[0]

known_name = input("Enter Student Name: ")

In [ ]:
cap = cv2.VideoCapture(0)

face_cascade = cv2.CascadeClassifier(
    cv2.data.haarcascades +
    'haarcascade_frontalface_default.xml'
)

In [ ]:
marked_attendance = []

while True:

    ret, frame = cap.read()

    if not ret:
        break

    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    faces = face_cascade.detectMultiScale(
        gray,
        1.3,
        5
    )

    face_locations = face_recognition.face_locations(
        rgb_frame
    )

    face_encodings = face_recognition.face_encodings(
        rgb_frame,
        face_locations
    )

    for (top, right, bottom, left), face_encoding in zip(
        face_locations,
        face_encodings
    ):

        matches = face_recognition.compare_faces(
            [known_encoding],
            face_encoding
        )

        name = "Unknown"

        if True in matches:
            name = known_name

        face = frame[top:bottom, left:right]

        try:

            # Emotion Detection
            result = DeepFace.analyze(
                face,
                actions=['emotion'],
                enforce_detection=False
            )

            emotion = result[0]['dominant_emotion']

            # Draw Rectangle
            cv2.rectangle(
                frame,
                (left, top),
                (right, bottom),
                (0, 255, 0),
                2
            )

            # Display Name + Emotion
            text = f"{name} - {emotion}"

            cv2.putText(
                frame,
                text,
                (left, top - 10),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.7,
                (255, 0, 0),
                2
            )

            # Attendance Marking
            if name not in marked_attendance:

                now = datetime.now()

                date = now.strftime('%Y-%m-%d')

                time = now.strftime('%H:%M:%S')

                new_entry = {
                    "Name": name,
                    "Date": date,
                    "Time": time,
                    "Emotion": emotion
                }

                attendance_df = pd.concat([
                    attendance_df,
                    pd.DataFrame([new_entry])
                ], ignore_index=True)

                marked_attendance.append(name)

                print(f"Attendance Marked for {name}")

        except Exception as e:
            print(e)

    cv2_imshow(frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

In [ ]:
attendance_df.to_csv(
    attendance_file,
    index=False
)

print("Attendance Saved Successfully")

In [ ]:
cap.release()

print("System Closed Successfully")